# Exploratory analysis (pynapple)

Quick look at both channels using pynapple - trial-structure tuning, raw example traces, and a population summary. Just getting a feel for the data, not a rigorous test of anything.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pynapple as nap

from CalciumKit.mouse import Mouse
from CalciumKit.trial_analysis import extract_trials

sns.set_style("ticks")
plt.rcParams.update({
    "font.family": "Arial",
    "font.size": 8,
    "axes.titlesize": 9,
    "axes.labelsize": 8,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "axes.linewidth": 1,
    "xtick.major.width": 1,
    "ytick.major.width": 1,
    "xtick.major.size": 2.5,
    "ytick.major.size": 2.5,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})


In [ ]:
raw_root = Path(r"D:\ImagingData\Raw")
processed_root = Path(r"D:\ImagingData\Processed\Chandler")

mouse = Mouse("Chandler", raw_root, processed_root)
session = mouse.sessions["day2"]
session.task = "tfc"
ts = session.trial_structure


## Trial-time tuning, both channels

Mean activity as a function of time since tone onset, pooled across all 5 trials, sorted by each cell's peak time - a quick population-tiling-style view.

In [ ]:
def trial_time_tuning(session, color, signal):
    tsd = session.tsd(color, signal)

    window_end = ts.shock_offset + ts.us_window  # CS+ / Trace / US only, no ITI

    trial_time_vals = np.full(len(tsd.t), np.nan)
    for tone in ts.tone_times:
        mask = (tsd.t >= tone - 5) & (tsd.t <= tone + window_end)
        trial_time_vals[mask] = tsd.t[mask] - tone
    valid = ~np.isnan(trial_time_vals)

    trial_time = nap.Tsd(t=tsd.t[valid], d=trial_time_vals[valid])
    trial_epochs = nap.IntervalSet(start=ts.tone_times - 5, end=ts.tone_times + window_end)

    n_bins = int(window_end + 5)  # ~1 bin/second
    return nap.compute_tuning_curves(data=tsd, features=trial_time, bins=n_bins, epochs=trial_epochs, return_pandas=True)


def plot_trial_tuning_heatmap(tc, title, cmap):
    tc_norm = (tc - tc.min()) / (tc.max() - tc.min())
    peak_order = tc_norm.idxmax(axis=0).sort_values().index
    tc_sorted = tc_norm[peak_order].T

    fig, ax = plt.subplots(figsize=(5, 3), dpi=300)
    im = ax.imshow(
        tc_sorted.values, aspect="auto", cmap=cmap,
        extent=[tc.index.min(), tc.index.max(), tc_sorted.shape[0], 0]
    )
    ax.axvline(0, color="black", linestyle="--", lw=1.2)
    ax.axvline(ts.shock_offset, color="#C62828", linestyle="--", lw=1.2)
    ax.set_yticks([0, tc_sorted.shape[0]])
    ax.set_yticklabels(["1", str(tc_sorted.shape[0])])
    ax.set_xlabel("Time from tone onset (s)")
    ax.set_ylabel("Cell (sorted by peak time)")
    ax.set_title(title, fontsize=13, fontweight="bold")
    plt.colorbar(im, ax=ax, label="Normalized activity", fraction=0.046, pad=0.04)
    sns.despine(ax=ax)
    plt.tight_layout()
    return fig, ax


In [ ]:
tc_red = trial_time_tuning(session, "red", "spks")
plot_trial_tuning_heatmap(tc_red, f"Neurons (n={tc_red.shape[1]})", "Reds")

tc_green = trial_time_tuning(session, "green", "dff")
plot_trial_tuning_heatmap(tc_green, f"Astrocytes (n={tc_green.shape[1]})", "Greens")


## Raw example traces, both channels

A few individual cells' raw signal over the first couple of trials, tone/trace/shock shaded.

In [ ]:
def plot_example_traces(session, color, signal, n_cells, main_color, window_s=350):
    tsd = session.tsd(color, signal)
    mask = tsd.t <= window_s
    t = tsd.t[mask]
    data = tsd.values[mask]

    cell_ids = range(min(n_cells, data.shape[1]))

    fig, axes = plt.subplots(len(list(cell_ids)), 1, figsize=(6, 1.2 * n_cells), dpi=300, sharex=True)
    for tone in ts.tone_times[ts.tone_times <= window_s]:
        for ax in axes:
            ax.axvspan(tone, tone + ts.tone_dur, color="#2E7D32", alpha=0.12, lw=0)
            ax.axvspan(tone + ts.tone_dur, tone + ts.tone_dur + ts.trace_dur, color="0.5", alpha=0.12, lw=0)
            ax.axvspan(tone + ts.shock_offset, tone + ts.shock_offset + 1, color="#C62828", alpha=0.3, lw=0)

    for i, ax in zip(cell_ids, axes):
        ax.plot(t, data[:, i], color=main_color, lw=0.8)
        ax.set_ylabel(f"cell {i}", fontsize=7, rotation=0, ha="right", va="center")
        sns.despine(ax=ax)

    axes[-1].set_xlabel("Time (s)")
    fig.suptitle(f"{color} / {signal}", fontsize=11, fontweight="bold")
    plt.tight_layout()
    return fig, axes


In [ ]:
plot_example_traces(session, "red", "spks", n_cells=4, main_color="#C62828")
plot_example_traces(session, "green", "dff", n_cells=4, main_color="#2E7D32")


## Cross-validated sorting - is the tiling real?

Sorting cells by their own peak time will always look like a clean diagonal, even from pure noise, since the sort itself creates the appearance of structure. The honest check: split trials in half, use one half to define the cell order, then plot the *other* half with that same order. If the diagonal survives on held-out trials, that's real sequential structure.

In [ ]:
def plot_cross_validated_heatmap(color, signal, cmap, label, epoch_window=None, seed=0):
    tsd = session.tsd(color, signal)
    window_end = ts.shock_offset + ts.us_window  # CS+ / Trace / US only, no ITI

    trial_data, t_rel = extract_trials(tsd.values, tsd.t, ts.tone_times, trial_window=window_end, window_before=5, zscore=True)

    if epoch_window is not None:
        mask = (t_rel >= epoch_window[0]) & (t_rel <= epoch_window[1])
        trial_data, t_rel = trial_data[:, :, mask], t_rel[mask]

    n_trials = trial_data.shape[1]
    rng = np.random.default_rng(seed)
    shuffled = rng.permutation(n_trials)
    half = n_trials // 2
    sort_trials, plot_trials = shuffled[:half], shuffled[half:]

    sort_avg = np.nanmean(trial_data[:, sort_trials, :], axis=1)
    plot_avg = np.nanmean(trial_data[:, plot_trials, :], axis=1)

    # normalize each cell using both halves pooled, so the two panels are directly comparable
    combined_min = np.minimum(sort_avg.min(axis=1), plot_avg.min(axis=1))[:, None]
    combined_max = np.maximum(sort_avg.max(axis=1), plot_avg.max(axis=1))[:, None]
    span = combined_max - combined_min
    span[span == 0] = 1
    sort_norm = (sort_avg - combined_min) / span
    plot_norm = (plot_avg - combined_min) / span

    order = np.argsort(np.argmax(sort_norm, axis=1))

    fig, axes = plt.subplots(1, 2, figsize=(7, 3), dpi=200)
    titles = [f"Sort half ({len(sort_trials)} trials) - defines order", f"Held-out half ({len(plot_trials)} trials) - same order"]
    for ax, data, title in zip(axes, [sort_norm[order], plot_norm[order]], titles):
        ax.imshow(data, aspect="auto", cmap=cmap, vmin=0, vmax=1, extent=[t_rel[0], t_rel[-1], data.shape[0], 0])
        ax.axvline(0, color="black", linestyle="--", linewidth=1.2)
        ax.axvline(ts.shock_offset, color="#C62828", linestyle="--", linewidth=1.2)
        ax.set_yticks([0, data.shape[0]])
        ax.set_yticklabels(["1", str(data.shape[0])])
        ax.set_title(title, fontsize=10, fontweight="bold")
        ax.set_xlabel("Time from tone onset (s)")
    axes[0].set_ylabel(f"{label} cell (sorted)", fontsize=11, fontweight="bold")
    plt.tight_layout()
    return fig, axes


plot_cross_validated_heatmap("red", "spks", "Reds", "Neuron")
plot_cross_validated_heatmap("green", "dff", "Greens", "Astrocyte")


### Restricted to specific epochs

The full-trial window is mostly quiet ITI, which dilutes any real tiling. Restricting the sort/held-out split to just the post-tone (CS+) or post-shock window tests for finer, epoch-specific structure where there's more time resolution relative to the window length.

In [ ]:
plot_cross_validated_heatmap("red", "spks", "Reds", "Neuron - post-tone", epoch_window=(0, ts.tone_dur))
plot_cross_validated_heatmap("green", "dff", "Greens", "Astrocyte - post-tone", epoch_window=(0, ts.tone_dur))

plot_cross_validated_heatmap("red", "spks", "Reds", "Neuron - post-shock", epoch_window=(ts.shock_offset, ts.shock_offset + 10))
plot_cross_validated_heatmap("green", "dff", "Greens", "Astrocyte - post-shock", epoch_window=(ts.shock_offset, ts.shock_offset + 10))


### Restricted to specific epochs

The full-trial window is mostly quiet ITI, which dilutes any real tiling. Restricting the sort/held-out split to just the post-tone (CS+) or post-shock window tests for finer, epoch-specific structure where there's more time resolution relative to the window length.

In [ ]:
plot_cross_validated_heatmap("red", "spks", "Reds", "Neuron - post-tone", epoch_window=(0, ts.tone_dur))
plot_cross_validated_heatmap("green", "dff", "Greens", "Astrocyte - post-tone", epoch_window=(0, ts.tone_dur))

plot_cross_validated_heatmap("red", "spks", "Reds", "Neuron - post-shock", epoch_window=(ts.shock_offset, ts.shock_offset + 10))
plot_cross_validated_heatmap("green", "dff", "Greens", "Astrocyte - post-shock", epoch_window=(ts.shock_offset, ts.shock_offset + 10))
